# Delete branches manually 

For young lungs (~PCW7), proximal pruning does not work as the lungs are too small and there is not a reliable difference in branches along the proximal-distal axis. However, because the lungs have such few branches, it is important to delete any spurious branches. For these young lungs, we must take a manual approach to ensure accuracy. We identify spurious branches by comparing the input and skeleton.  
Before running this notebook, run six_plot_loops and identify artifact branch nodes. Select the artifact branches at the distal node, and note node IDs. 

In [2]:
import pandas as pd 
import cv2
import tifffile as tf
import numpy as np
import pandas as pd
import re
from skimage.morphology import skeletonize
import skan 
import os
# conda install -c conda-forge napari pyside6
import napari

In [ ]:
folder_path = r"path\to\08_Proximal_pruning\input\\"
skeleton_img = tf.imread(folder_path + "skeleton.tif") 
input_img = tf.imread(r"path\to\03_Mesh_to_Image\output\\" + "EH3669LL_7um.tif")
voxel = int(7)
# First compare the matlab graph with the image data 
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(input_img, name = ('input_img'), opacity = 0.4)
viewer.add_image(skeleton_img, name=('skeleton_img'))

# IF there are artifacts, note the end nodes of the spurious branches and input node IDs in the next cell
# If there are NO artifacts, go directly fo notebook 9. 

<Image layer 'skeleton_img' at 0x29ab13fe890>

In [ ]:
delete_nodes = (54, 55) 
node_positions = pd.read_csv(r"path\to\06_Plot_loop\output\\" + "node_positions.csv", delimiter = ",", names=["Node_ID", "Y", "X", "Z"])

In [ ]:
import numpy as np
import tifffile as tf
from scipy.ndimage import convolve
import os

######### 1. Build the suppression mask #########

# Collect all coordinates of branches to delete
delete_mask = np.zeros_like(binary_skeleton, dtype=bool)

for path_id in matched_path_ids:
    path_coords = skeleton.path_coordinates(int(path_id))
    for coord in path_coords:
        z, y, x = int(coord[0]), int(coord[1]), int(coord[2])
        delete_mask[z, y, x] = True

######### 2. Identify junction nodes to preserve #########
# A junction pixel has degree >= 3 (connected to 3+ neighbors)
junction_img = skan.csr.make_degree_image(binary_skeleton)
junction_mask = junction_img >= 3

# Do not delete junction pixels
delete_mask = delete_mask & ~junction_mask

######### 3. Apply suppression #########
corrected = (np.copy(binary_skeleton) > 0).astype(np.uint8) * 255
corrected[delete_mask] = 0

######### 4. Clean up residual stubs #########
# After suppression, isolated pixels or mini-branches
# may remain near junctions -> prune them iteratively

def prune_endpoints(skel_3d, iterations=3):
    """
    Iteratively removes terminal pixels (endpoints)
    to eliminate residual stubs.
    An endpoint = skeleton pixel with only 1 active neighbor.
    """
    kernel = np.ones((3, 3, 3), dtype=np.uint8)
    kernel[1, 1, 1] = 0  # Do not count the pixel itself

    result = skel_3d.copy()
    for i in range(iterations):
        binary = (result > 0).astype(np.uint8)
        neighbor_count = convolve(binary, kernel, mode='constant', cval=0)
        # Endpoint: active pixel with exactly 1 active neighbor
        endpoints = (result > 0) & (neighbor_count == 1)
        if not np.any(endpoints):
            print(f"  Cleanup done in {i} iteration(s)")
            break
        result[endpoints] = 0
    return result

print("Cleaning up residual stubs...")
corrected = prune_endpoints(corrected, iterations=5)
corrected = (corrected > 0).astype(np.uint8) * 255

######### 5. Save #########
output_path = os.path.join(folder_path, "corrected_skeleton.tif")
tf.imwrite(output_path, corrected)
print(f"Corrected skeleton saved: {output_path}")

# Quick check
n_before = int(np.sum(binary_skeleton > 0))
n_after  = int(np.sum(corrected > 0))
print(f"Pixels before: {n_before} | after: {n_after} | removed: {n_before - n_after}")